# Apoyo simbólico con SymPy para la plataforma antivibratoria

Este notebook se deja como respaldo algebraico del proyecto. La idea no es solo mostrar fórmulas finales, sino dejar trazado un flujo útil de trabajo:

1. planteamiento del modelo nominal completo de tercer orden,
2. obtención de la función de transferencia por dos caminos,
3. análisis del polinomio característico y condición de Routh,
4. reducción física por dinámica eléctrica rápida,
5. modelo reducido usado para la sintonía,
6. obtención del controlador IMC,
7. cálculo simbólico de los parámetros del PID ideal usando `solve`,
8. relación entre seguimiento y rechazo de perturbaciones,
9. sustitución numérica del caso del proyecto.

La notación se dejó consistente con el informe: el parámetro de sintonía de IMC se denota por $\lambda$.

In [ ]:
import sympy as sp
sp.init_printing(use_unicode=True)
sp.__version__

## 1. Símbolos y variables

In [ ]:
m, c, k = sp.symbols('m c k', positive=True, nonzero=True)
K_f, K_e = sp.symbols('K_f K_e', positive=True, nonzero=True)
L, R = sp.symbols('L R', positive=True, nonzero=True)
s, lambda_ = sp.symbols('s lambda', positive=True, nonzero=True)

q, qd, qdd = sp.symbols('q qd qdd')
u, i = sp.symbols('u i')
zdd = sp.symbols('zdd')

Q, U, I, D = sp.symbols('Q U I D')
Kc, tau_I, tau_D = sp.symbols('K_c tau_I tau_D', positive=True, nonzero=True)

m, c, k, K_f, K_e, L, R, s, lambda_

## 2. Modelo nominal completo de tercer orden

El modelo físico completo alrededor del punto nominal, en coordenada relativa $q=x-z$, es:

$$
m\ddot q + c\dot q + kq = K_f i - m\ddot z,
$$

$$
L\dot i + Ri + K_e\dot q = u.
$$

La planta nominal para análisis se obtiene anulando temporalmente la perturbación de base, es decir, tomando $\ddot z = 0$.

In [ ]:
eq_mech = sp.Eq(m*qdd + c*qd + k*q, K_f*i - m*zdd)
eq_elec = sp.Eq(L*sp.Symbol('idot') + R*i + K_e*qd, u)

eq_mech, eq_elec

## 3. Representación en espacio de estados

Tomando los estados $x_1=q$, $x_2=\dot q$ y $x_3=i$, el modelo nominal queda:

$$
\dot x = Ax + Bu, \qquad y = Cx + Du.
$$

In [ ]:
A = sp.Matrix([
    [0, 1, 0],
    [-k/m, -c/m, K_f/m],
    [0, -K_e/L, -R/L],
])
B = sp.Matrix([[0], [0], [1/L]])
C = sp.Matrix([[1, 0, 0]])
Dmat = sp.Matrix([[0]])

A, B, C, Dmat

## 4. Función de transferencia desde espacio de estados

Aquí se verifica simbólicamente la relación

$$
G(s)=C(sI-A)^{-1}B + D.
$$

In [ ]:
I3 = sp.eye(3)
G_ss = sp.simplify((C * (s*I3 - A).inv() * B + Dmat)[0])
G_ss

## 5. Función de transferencia obtenida resolviendo el sistema en Laplace

Ahora se llega a la misma expresión resolviendo directamente las ecuaciones en el dominio de Laplace. Esta parte sí usa `solve`, de modo que el notebook tenga una utilidad real y no sea solo una hoja de fórmulas copiadas.

In [ ]:
eq1 = sp.Eq((m*s**2 + c*s + k)*Q, K_f*I - m*D)
eq2 = sp.Eq((L*s + R)*I + K_e*s*Q, U)

sol_full = sp.solve((eq1, eq2), (Q, I), dict=True)[0]
sol_full

In [ ]:
G_u_from_solve = sp.simplify(sol_full[Q].subs(D, 0) / U)
G_d_from_solve = sp.simplify(sol_full[Q].subs(U, 0) / D)

G_u_from_solve, G_d_from_solve

In [ ]:
G_nominal = sp.simplify(K_f / ((L*s + R)*(m*s**2 + c*s + k) + K_f*K_e*s))
sp.simplify(G_ss - G_nominal)

Si la última salida da cero, significa que la función de transferencia obtenida desde espacio de estados coincide exactamente con la obtenida al resolver las ecuaciones.

In [ ]:
den_full = sp.expand(sp.denom(G_nominal))
num_full = sp.expand(sp.numer(G_nominal))
num_full, den_full

## 6. Transferencia de la perturbación de base a la salida

Como el proyecto está formulado como rechazo de perturbaciones, también conviene dejar explícita la relación desde la perturbación $D(s)=\ddot Z(s)$ hasta la salida relativa $Q(s)$:

In [ ]:
Gd_full = sp.simplify(G_d_from_solve)
Gd_full

## 7. Polinomio característico y condición de Routh-Hurwitz

Del denominador del sistema completo se obtiene el polinomio característico:

$$
a_3s^3 + a_2s^2 + a_1s + a_0,
$$

con

$$
a_3=Lm,\quad a_2=Lc+Rm,\quad a_1=Lk+Rc+K_fK_e,\quad a_0=Rk.
$$

In [ ]:
a3 = sp.simplify(L*m)
a2 = sp.simplify(L*c + R*m)
a1 = sp.simplify(L*k + R*c + K_f*K_e)
a0 = sp.simplify(R*k)

a3, a2, a1, a0

In [ ]:
routh_s1 = sp.simplify((a2*a1 - a3*a0)/a2)
cond_routh = sp.factor(a2*a1 - a3*a0)

routh_s1, cond_routh

El término `cond_routh` es el que debe ser positivo para completar la condición de estabilidad del sistema cúbico.

## 8. Reducción física usando dinámica eléctrica rápida

Si la dinámica eléctrica es mucho más rápida que la dinámica mecánica dominante, se usa la aproximación cuasiestática

$$
L\dot i \approx 0.
$$

Entonces se resuelve algebraicamente la corriente y se sustituye en la ecuación mecánica.

In [ ]:
i_qs = sp.solve(sp.Eq(R*i + K_e*qd, u), i)[0]
i_qs

In [ ]:
eq_reduced_physical = sp.Eq(m*qdd + c*qd + k*q, K_f*i_qs)
eq_reduced_physical_simplified = sp.Eq(
    m*qdd + (c + K_f*K_e/R)*qd + k*q,
    (K_f/R)*u
)

eq_reduced_physical, eq_reduced_physical_simplified

In [ ]:
G_reduced_physical = sp.simplify((K_f/R) / (m*s**2 + (c + K_f*K_e/R)*s + k))
G_reduced_physical

## 9. Modelo reducido de sintonía usado en el proyecto

Para simplificar la síntesis del IMC-PID, el proyecto adopta el modelo de diseño

$$
G_r(s)=\frac{1}{ms^2+cs+k}.
$$

Este modelo no reemplaza la planta completa; solo se usa como base para la sintonía.

Sin embargo, como el problema central del proyecto es rechazo de perturbaciones, también conviene escribir explícitamente la versión forzada:

$$
m\ddot q + c\dot q + kq = u + f_p(t),
$$

donde $f_p(t)$ representa una fuerza de perturbación equivalente. Si se desea conectar esta expresión con el modelo completo, puede interpretarse como una representación concentrada del efecto de la base vibrante.

In [ ]:
G_tune = sp.simplify(1 / (m*s**2 + c*s + k))
omega_n = sp.simplify(sp.sqrt(k/m))
zeta = sp.simplify(c / (2*sp.sqrt(m*k)))

G_tune, omega_n, zeta

In [ ]:
F_p = sp.symbols('F_p')
Q_reduced_forced = sp.simplify(G_tune*U + G_tune*F_p)
G_force_reduced = sp.simplify(Q_reduced_forced / F_p).subs(U, 0)

Q_reduced_forced, G_force_reduced

## 10. Filtro IMC y controlador resultante

Se toma el filtro IMC más simple consistente con la formulación usada en el informe:

$$
F(s)=\frac{1}{\lambda s+1}.
$$

Y se calcula simbólicamente

$$
C(s)=\frac{F(s)}{G_r(s)\left(1-F(s)\right)}.
$$

In [ ]:
F = sp.simplify(1 / (lambda_*s + 1))
C_imc = sp.simplify(F / (G_tune * (1 - F)))
C_imc

In [ ]:
C_imc_expanded = sp.expand(C_imc)
C_imc_expanded

## 11. Obtención de los parámetros del PID ideal usando `solve`

El PID ideal se escribe como

$$
C_{PID}(s)=K_c\left(1+\frac{1}{\tau_I s}+\tau_D s\right).
$$

Para comparar coeficientes de manera limpia, se multiplica por $s$ en ambos lados y luego se usa `solve`.

In [ ]:
C_pid_ideal = sp.expand(Kc * (1 + 1/(tau_I*s) + tau_D*s))
lhs = sp.expand(s * C_imc)
rhs = sp.expand(s * C_pid_ideal)

lhs, rhs

In [ ]:
poly_lhs = sp.Poly(lhs, s)
poly_rhs = sp.Poly(rhs, s)

eqs_pid = [
    sp.Eq(poly_lhs.coeff_monomial(s**2), poly_rhs.coeff_monomial(s**2)),
    sp.Eq(poly_lhs.coeff_monomial(s), poly_rhs.coeff_monomial(s)),
    sp.Eq(poly_lhs.coeff_monomial(1), poly_rhs.coeff_monomial(1)),
]

eqs_pid

In [ ]:
sol_pid = sp.solve(eqs_pid, (Kc, tau_I, tau_D), dict=True)[0]
sol_pid

El resultado simbólico debe dar:

$$
K_c=\frac{c}{\lambda},\qquad \tau_I=\frac{c}{k},\qquad \tau_D=\frac{m}{c}.
$$

## 12. Paso a forma paralela

A partir del PID ideal también se puede obtener la forma paralela:

$$
K_p = K_c,\qquad K_i = \frac{K_c}{\tau_I},\qquad K_d = K_c\tau_D.
$$

In [ ]:
Kp_expr = sp.simplify(sol_pid[Kc])
Ki_expr = sp.simplify(sol_pid[Kc] / sol_pid[tau_I])
Kd_expr = sp.simplify(sol_pid[Kc] * sol_pid[tau_D])

Kp_expr, Ki_expr, Kd_expr

## 13. Relación con el bloque PID filtrado de Simulink

El bloque suele escribirse como

$$
C_b(s)=P\left(1+\frac{1}{Is}+D\frac{N}{1+N/s}\right).
$$

Aquí se verifica que el término derivativo filtrado tiende al derivativo ideal cuando $N\to\infty$.

In [ ]:
Pblk, Iblk, Dblk, N = sp.symbols('P I D N', positive=True, nonzero=True)
deriv_filtered = sp.simplify(Dblk * N / (1 + N/s))
sp.simplify(deriv_filtered), sp.limit(deriv_filtered, N, sp.oo)

## 14. Seguimiento y rechazo de perturbaciones

Con el modelo reducido y el controlador IMC, se pueden escribir tres relaciones cerradas útiles:

- seguimiento: $T_r(s)=\dfrac{C(s)G_r(s)}{1+C(s)G_r(s)}$
- sensibilidad: $S(s)=\dfrac{1}{1+C(s)G_r(s)}$
- perturbación forzada al modelo reducido: $\dfrac{Q(s)}{F_p(s)}=\dfrac{G_r(s)}{1+C(s)G_r(s)}$

Estas expresiones ayudan a separar el caso de seguimiento del caso de regulación frente a perturbaciones. En particular, la última es la relevante cuando el setpoint es fijo y lo que se desea es que la salida no se mueva ante una excitación externa.

In [ ]:
L_open = sp.simplify(C_imc * G_tune)
T_tracking = sp.simplify(L_open / (1 + L_open))
S_sensitivity = sp.simplify(1 / (1 + L_open))
T_force = sp.simplify(G_tune / (1 + L_open))

T_tracking, S_sensitivity, T_force

## 15. Caso numérico del proyecto

Se sustituyen los valores usados en el informe:

$$
m=1,\qquad c=5,\qquad k=100,\qquad \lambda=0.1.
$$

In [ ]:
vals = {
    m: 1,
    c: 5,
    k: 100,
    lambda_: sp.Rational(1, 10),
}

G_tune_num = sp.simplify(G_tune.subs(vals))
C_imc_num = sp.simplify(C_imc.subs(vals))
sol_pid_num = {sym: sp.simplify(expr.subs(vals)) for sym, expr in sol_pid.items()}
Kp_num = sp.simplify(Kp_expr.subs(vals))
Ki_num = sp.simplify(Ki_expr.subs(vals))
Kd_num = sp.simplify(Kd_expr.subs(vals))

G_tune_num, C_imc_num, sol_pid_num, Kp_num, Ki_num, Kd_num

In [ ]:
omega_n_num = sp.simplify(omega_n.subs(vals))
zeta_num = sp.simplify(zeta.subs(vals))

omega_n_num, zeta_num

## 16. Resumen automático útil

La siguiente celda deja un pequeño resumen que sirve como chequeo rápido antes de copiar resultados al informe.

In [ ]:
summary = {
    'Planta nominal de tercer orden': G_nominal,
    'Transferencia de perturbacion': Gd_full,
    'Modelo reducido fisico': G_reduced_physical,
    'Modelo de sintonia': G_tune_num,
    'Transferencia reducida de fuerza de perturbacion': G_force_reduced,
    'Control IMC numerico': sp.expand(C_imc_num),
    'Lazo cerrado perturbacion-salida': T_force,
    'Kc': sol_pid_num[Kc],
    'tau_I': sol_pid_num[tau_I],
    'tau_D': sol_pid_num[tau_D],
    'Kp': Kp_num,
    'Ki': Ki_num,
    'Kd': Kd_num,
    'omega_n': omega_n_num,
    'zeta': zeta_num,
}

summary